# Regressão logística sobre as listas da MobileNetV3 

Este notebook usa as listas de 960 números por foto guardadas pelo notebook `extracao-features-mobilenetv3-large`. A rede **não roda mais aqui**: as listas já estão prontas, e o que falta é ajustar a regressão logística em cima delas.

Fluxo: lista de números → regressão (com pesos) → score por rótulo → corte → alerta.


## 1. Ambiente

In [ ]:
import numpy as np
import pandas as pd

print("numpy:", np.__version__)
print("pandas:", pd.__version__)

# Os arquivos vêm do notebook de extração: a saída dele aparece dentro de /kaggle/input/
PASTA = "/kaggle/input/notebooks/joyceximenes/extracao-features-mobilenetv3-large/"

## 2. Carregar as listas e o manifesto

`features_large.npy` tem uma linha por foto (as 960 médias dos mapas da rede). `features_manifest.csv` diz, na mesma ordem, de que conjunto é cada foto e quais são os 4 rótulos dela.

In [ ]:
# Etapa 2: carregar. 
# X = as listas (uma linha por foto, 960 colunas); manifest = o que se sabe de cada foto
X = np.load(PASTA + "features_large.npy")
manifest = pd.read_csv(PASTA + "features_manifest.csv")

# os 4 rótulos, na ordem do cartão. 
# 1 = a foto tem o problema, 0 = não tem (o gabarito dos anotadores)
ROTULOS = ["BLR", "BRT", "DRK", "FRM"]

print("X:", X.shape, X.dtype)
print("manifest:", manifest.shape, "| colunas:", list(manifest.columns))

# conferências: a linha i de X tem de ser a foto da linha i do manifesto, 
# senão os rótulos ficam trocados
assert X.shape == (len(manifest), 960), "X e manifesto com tamanhos diferentes"
assert (manifest["i"].to_numpy() == np.arange(len(manifest))).all(), "a coluna i do manifesto não está em ordem"
assert np.isfinite(X).all(), "há NaN ou infinito em X"
print("X e manifesto casam: a linha i de X é a foto i do manifesto")

## 3. Separar treino e validação

O treino é o conjunto de onde se aprendem os pesos; a validação só mede. As quatro peças abaixo são as que o resto do notebook vai usar.

In [ ]:
# Etapa 3: separar. O manifesto diz de que conjunto cada foto é. 
eh_treino = (manifest["split"] == "train").to_numpy()
eh_val = (manifest["split"] == "val").to_numpy()

# as quatro peças do desenho: as listas e os rótulos de cada conjunto
X_treino = X[eh_treino]
y_treino = manifest.loc[eh_treino, ROTULOS].to_numpy()
X_val = X[eh_val]
y_val = manifest.loc[eh_val, ROTULOS].to_numpy()

print("X_treino:", X_treino.shape, "| y_treino:", y_treino.shape)
print("X_val:   ", X_val.shape, "| y_val:   ", y_val.shape)


assert eh_treino.sum() + eh_val.sum() == len(manifest) and not (eh_treino & eh_val).any()

# que parte das fotos tem cada problema, em cada conjunto: 
# dá a ideia do desbalanceamento antes de ajustar
fracao = pd.DataFrame({"treino": y_treino.mean(axis=0), "validação": y_val.mean(axis=0)}, index=ROTULOS)
print()
print("% de fotos com o rótulo (1):")
print((fracao * 100).round(1))

o conjunto se encontra balanceado entre treino e validação

## 4. Regressão logística


um rótulo por vez. primeiro, o blr

In [ ]:
[i for i, rotulo in enumerate(ROTULOS) if rotulo == "BLR"]

BLR está no índice 0.

In [ ]:
blr_treino = y_treino[:,0]
blr_val = y_val[:,0]

print(blr_treino.shape)
print(blr_val.shape)

In [ ]:
# quantidade de '1' em blr treino e validação
print((blr_treino == 1).sum())
print((blr_val == 1).sum())

**Explicação — passo 1: o gabarito do BLR**

A regressão logística trata **um rótulo por vez**. Por isso pegamos só a coluna do BLR de `y_treino` e de `y_val`: uma lista com um 0 ou 1 por foto.

```
y_treino (23431 × 4)         blr_treino (23431,)
 BLR BRT DRK FRM
  1   1   0   0                 1
  1   0   0   1        →        1
  0   0   0   1                 0
  ▲ só esta coluna
```

- `y_treino[:, 0]`: antes da vírgula vão as linhas (`:` quer dizer todas) e depois a coluna. A coluna 0 é o BLR, a posição dele em `ROTULOS`.
- 9.552 das 23.431 fotos de treino (40,8%) têm BLR, e 3.279 das 7.750 de validação (42,3%). São as frações da tabela da etapa 3.

criar regressão e ajustar os pesos 

In [ ]:
from sklearn.linear_model import LogisticRegression

# criar modelo
modelo = LogisticRegression()

# ajuste com a listas e gabarito de treino
modelo.fit(X_treino, blr_treino)

# imprimir forma dos pesos e viés
print(modelo.coef_.shape) # pesos
print(modelo.intercept_.shape) # viés

**Explicação — passo 2: aprender os pesos**

Esta célula é o momento de **aprender** da aula.

```
X_treino (23431 × 960) ─┐
                        ├─▶ .fit ─▶ modelo: 960 pesos + 1 viés
blr_treino (23431,)  ───┘
```

- `LogisticRegression()` cria o modelo vazio, ainda sem pesos (como o `norm` na extração).
- `.fit` repete o ciclo da aula: soma ponderada das 960 listas → sigmoide → score, compara com o gabarito e ajusta os pesos. Só o treino participa.
- `coef_` tem forma `(1, 960)`: um peso para cada número da lista. `intercept_` tem forma `(1,)`: o viés.


In [ ]:
probas = modelo.predict_proba(X_val)
# prob devolve [chance de 0, chance de 1]

scores_blr = probas[:, 1] # pega a coluna com chance de ser 1 (fica no indice 1)

print(scores_blr.shape)
print(scores_blr.min())
print(scores_blr.max())
print(scores_blr.mean())
print(scores_blr[:5])

**Explicação — passo 3: aplicar nos dados de validação**

Esta célula é o momento de **aplicar**: os pesos aprendidos no treino entram na conta com as fotos de validação, que o modelo nunca viu.

```
foto da validação → 960 números → soma ponderada com os pesos → sigmoide → score (0 a 1)
```

- `predict_proba` devolve 2 colunas por foto: `[chance de 0, chance de 1]`. O score do BLR é a coluna 1, `probas[:, 1]`.
- A forma `(7750,)` mostra um score para cada foto da validação.

**As 5 primeiras fotos de validação:**

| foto | score (chance de BLR) | gabarito |
|---|---|---|
| 0 | 0,05 | 0 |
| 1 | 0,22 | 0 |
| 2 | 0,26 | 1 |
| 3 | 0,42 | 1 |
| 4 | 0,53 | 0 |

O score não é um veredito. Com um corte em 0,5, as fotos 2 e 3 (que têm BLR) ficariam sem alerta, e a foto 4 (que não tem) receberia alerta. O corte é decidido depois, como no s5a.

**Os números impressos**
- Média 0,408: fica perto da fração real de fotos com BLR na validação (0,423). É um bom sinal de que o modelo não está alto ou baixo demais no geral, mas não prova que ele acerta foto a foto.
- Mínimo 0,0003 e máximo 0,9999: para algumas fotos o modelo dá uma chance quase certa. Isso não garante que ele acerte nelas, porque o score estima a chance, não é taxa de acerto.

Cinco fotos não dizem se o modelo é bom. Falta medir nas 7.750 (passo 4).

In [ ]:
from sklearn.metrics import roc_auc_score

# gabarito, scores
auc_blr = roc_auc_score(blr_val,scores_blr)

print(f"AUC na regressão: {auc_blr:2f}")
print(f"numero de iterações: {modelo.n_iter_}")

na heuristica, o valor era de 0,789. houve um aumento

**Explicação — passo 4: medir com a ROC-AUC**

Até aqui o modelo só gerou scores. Agora eles são comparados com o gabarito da validação, que o modelo nunca viu.

```
scores das fotos SEM BLR:  ·· ··· ····· ··  ·                    (tendem a ficar perto de 0)
scores das fotos COM BLR:               · ·· ···· ·· ····        (tendem a ficar perto de 1)
                          0 ─────────────────────────────────── 1
```

A **AUC** é a chance de, sorteando uma foto com BLR e uma sem, a que tem BLR receber o score maior. 0,5 é sorteio e 1,0 é ordenar sempre certo. Ela mede a **ordem** dos scores, então não precisa de corte.

- `roc_auc_score(blr_val, scores_blr)`: o primeiro argumento é o gabarito e o segundo, os scores. A nota sai da validação, e o treino não entra.
- A heurística (linha de base do Sprint 4) tinha AUC 0,789, com intervalo de confiança de 95% de [0,779; 0,799]. A nota desta saída fica em torno de 0,85, acima de todo esse intervalo. Por isso o aumento que você anotou é maior que a margem de erro da heurística.

**Cuidados antes de afirmar o ganho**
1. 248 das 7.750 fotos de validação têm cópia exata no treino, o que pode deixar a nota um pouco otimista. O efeito ainda não foi medido.
2. É o primeiro modelo, ainda com o aviso de convergência.
3. Falta o intervalo de confiança do próprio modelo (bootstrap pareado com a heurística, nas mesmas fotos).

Próximo: testar mais repetições e a escala das listas, para ver se a nota muda.

ajuste com o parametro max_iter

In [ ]:
modelo2 = LogisticRegression(max_iter=1000)

modelo2.fit(X_treino, blr_treino)

# scores de validação
probas2 = modelo2.predict_proba(X_val)

scores2_blr = probas2[:, 1]

auc2_blr = roc_auc_score(blr_val,scores2_blr)

print(f"AUC na regressão com max_iter = 1000: {auc2_blr:2f}")
print(f"numero de iterações: {modelo2.n_iter_}")

**Explicação — passo 5: deixar o ajuste terminar (`max_iter`)**

O aviso do passo 2 dizia que o ajuste parou no limite de 100 repetições. Aqui o limite sobe para 1000, e o resto fica igual. Os nomes novos (`modelo2`, `scores2_blr`, `auc2_blr`) preservam o primeiro modelo e a primeira nota, para comparar.

| | limite | repetições usadas | aviso | AUC na validação |
|---|---|---|---|---|
| `modelo` | 100 (padrão) | 100 | sim | 0,846007 |
| `modelo2` | 1000 | 136 | não | 0,845823 |

```
0           100   136                             1000
├────────────┤▒▒▒▒▒┤·································┤
  modelo parou aqui ┘   └ modelo2 parou sozinho
```

**O que isso mostra**
- `max_iter` é só o **limite**. O `modelo2` não gastou as 1000: parou sozinho na repetição 136, quando o erro deixou de cair. O limite de 100 era curto por 36 repetições.
- O aviso sumiu, porque agora o ajuste terminou de verdade.
- A nota quase não mudou: a diferença é de 0,0002. O aviso de convergência não quer dizer "modelo ruim". Aqui o ajuste já estava quase no fim.
- Olhar a validação neste teste é aceitável: ele só deixa o ajuste terminar, e não escolhe o melhor entre vários valores.

Daqui em diante vale usar o `modelo2`, que terminou de ajustar.



escalar as listas

In [ ]:
from sklearn.preprocessing import StandardScaler

# scaler com dados de treino
scaler = StandardScaler()
scaler.fit(X_treino)

# transformar treino e validação - colocar os valores na mesma regra usando a escala
X_treino_esc = scaler.transform(X_treino)
X_val_esc = scaler.transform(X_val)

# modelo com essa transformação
modelo3 = LogisticRegression(max_iter=1000)

modelo3.fit(X_treino_esc, blr_treino)

# scores de validação
probas3 = modelo3.predict_proba(X_val_esc)

scores3_blr = probas3[:, 1]

auc3_blr = roc_auc_score(blr_val,scores3_blr)

print(f"AUC na regressão com max_iter = 1000 e em escala: {auc3_blr:.3f}")
print(f"numero de iterações: {modelo3.n_iter_}")

**Explicação — passo 6: escalar as listas**

As colunas de `X` são os 960 números da lista de cada foto: cada uma é um padrão da última camada de filtros da rede, e o valor diz quanto dele a rede achou na foto. Nenhuma coluna tem nome.

Elas têm réguas diferentes: no treino, o desvio-padrão vai de 0,05 (coluna 877) a 1,31 (coluna 770). Escalar põe todas na mesma régua. Para cada coluna:

```
(valor − média da coluna) ÷ desvio da coluna
```

Exemplo de brinquedo (4 fotos, 2 colunas):

```
              antes                    depois da régua
coluna A    0,1  0,2  0,1  0,15     −0,90  1,51  −0,90  0,30
coluna B    2    8    11   5        −1,34  0,45   1,34 −0,45
```

**A regra: a régua é medida só no treino.** A média e o desvio são aprendidos dos dados, como os pesos. A validação só usa a régua que o treino montou:

```
X_treino ──fit──▶ régua (960 médias + 960 desvios)
X_treino ──transform──▶ X_treino_esc
X_val    ──transform──▶ X_val_esc      (régua do treino, não da validação)
```

**Resultado, com o mesmo limite de 1000 repetições do `modelo2`:**

| | escala | repetições usadas | aviso | AUC (3 casas) |
|---|---|---|---|---|
| `modelo` | não | 100 (limite 100) | sim | 0,846 |
| `modelo2` | não | 136 | não | 0,846 |
| `modelo3` | sim | 62 | não | 0,845 |

**Como ler**
- A escala fez o ajuste terminar em **menos da metade das repetições** (62 contra 136).
- A nota ficou praticamente igual. A diferença na terceira casa é da ordem de 0,001, bem menor que a margem de erro desta validação (o intervalo de confiança da heurística tem cerca de ±0,01).
- Os dois caminhos que o aviso apontava resolvem o problema sem mudar a nota. A escala é a mais segura daqui em diante: ela ajuda o ajuste e será importante quando entrarmos na regularização.

Para comparar diferenças tão pequenas, vale imprimir a AUC com mais casas (`.4f`).

**Passo 7 — os outros 3 rótulos** 

Uma regressão logística por rótulo, com a mesma receita do passo 6: escala e `max_iter=1000`. A régua (`X_treino_esc` e `X_val_esc`) é a que já foi medida só no treino. O laço refaz também o BLR, como conferência: ele tem de dar a mesma AUC do `modelo3`.

In [ ]:
# Passo 7: os 4 rótulos, uma regressão para cada um, com a mesma receita do passo 6 (escala + max_iter=1000).
# X_treino_esc e X_val_esc já existem: a régua foi medida só no treino, no passo 6

# AUC da heurística na validação (linha de base do Sprint 4, notebook baseline), para comparar
AUC_HEURISTICA = {"BLR": 0.789, "BRT": 0.833, "DRK": 0.822, "FRM": 0.542}

linhas = []
scores_val = {}  # os scores de validação de cada rótulo, para os próximos passos

for j, rotulo in enumerate(ROTULOS):
    # o gabarito deste rótulo: a coluna j de y_treino e de y_val (como no passo 1)
    gabarito_treino = y_treino[:, j]
    gabarito_val = y_val[:, j]

    # aprender: só com o treino
    modelo_rotulo = LogisticRegression(max_iter=1000)
    modelo_rotulo.fit(X_treino_esc, gabarito_treino)

    # aplicar: a coluna 1 do predict_proba é a chance de o rótulo ser 1
    scores_val[rotulo] = modelo_rotulo.predict_proba(X_val_esc)[:, 1]

    # medir: AUC na validação
    auc = roc_auc_score(gabarito_val, scores_val[rotulo])
    linhas.append(
        {
            "rótulo": rotulo,
            "AUC heurística": AUC_HEURISTICA[rotulo],
            "AUC modelo": auc,
            "diferença": auc - AUC_HEURISTICA[rotulo],
            # menor que 1000 = o ajuste terminou sozinho, sem bater no limite
            "repetições": int(modelo_rotulo.n_iter_[0]),
        }
    )

tabela = pd.DataFrame(linhas)
print(tabela.round(4).to_string(index=False))

# conferência: o BLR refeito no laço tem de dar a mesma AUC do passo 6 (mesmos dados, mesma receita)
assert abs(tabela.loc[tabela["rótulo"] == "BLR", "AUC modelo"].iloc[0] - auc3_blr) < 1e-9
print("conferência: o BLR do laço é igual ao modelo3 do passo 6")

**Explicação — passo 7: um modelo por rótulo**

O laço repete, para cada rótulo, as mesmas três coisas do passo a passo: aprender no treino, aplicar na validação e medir.

```
para cada rótulo (BLR, BRT, DRK, FRM):
    gabarito = coluna do rótulo em y_treino e em y_val
    treino ──fit──▶ pesos deste rótulo      (960 pesos + 1 viés)
    validação ──▶ scores ──▶ AUC
```

São 4 modelos independentes, com 4 × 961 = 3.844 parâmetros no total. Os 4 scores de uma foto não se somam a 1, porque o problema é multirrótulo, como na aula. O BLR refeito no laço deu a mesma AUC do `modelo3`, o que confere o laço.

**Resultado (validação, 7.750 fotos)**

| rótulo | AUC da heurística | IC 95% da heurística | AUC do modelo | repetições |
|---|---|---|---|---|
| BLR | 0,789 | [0,779; 0,799] | 0,8455 | 62 |
| BRT | 0,833 | [0,813; 0,853] | 0,8450 | 68 |
| DRK | 0,822 | [0,800; 0,842] | 0,8755 | 66 |
| FRM | 0,542 | [0,528; 0,554] | 0,7687 | 65 |

**Como ler**
- Todos os ajustes terminaram sozinhos, em cerca de 65 repetições, bem abaixo do limite de 1000.
- **FRM** teve o maior salto, de 0,542 (quase acaso) para 0,769. A lista da rede traz informação de enquadramento que o `edgeScore` não captava.
- **BLR e DRK** ficam acima de todo o intervalo de confiança da heurística.
- **BRT**: 0,845 fica **dentro** do intervalo da heurística. Só com esta tabela, não dá para afirmar ganho no BRT.

**Cuidados**
1. Comparar com o intervalo da heurística é uma leitura aproximada. O rigoroso é o bootstrap pareado, nas mesmas fotos.
2. A AUC mostra pouco do desempenho em rótulos raros: BRT e DRK estão em cerca de 6% das fotos. Falta a precisão média (AP).
3. 248 fotos de validação têm cópia exata no treino, o que pode deixar as notas um pouco otimistas.
4. Ainda não há corte nem regularização: são os modelos mais simples.

AP - average precision - precisão media

mede se o score põe no topo da lista as fotos que têm o problema.

In [ ]:
# AP - blr
from sklearn.metrics import average_precision_score

aps_blr = average_precision_score(blr_val,scores_val["BLR"])

print(f"AP: {aps_blr:.4f}")

**Explicação — passo 8: precisão média (AP)**

A AP mede se o score põe **no topo da lista** as fotos que têm o problema. As fotos são ordenadas do maior score para o menor, e em cada foto com o problema (✓) olha-se a precisão até ali: das fotos vistas, que fração tem o problema? A AP é a média dessas precisões.

Exemplo de brinquedo (10 fotos, 4 com BLR):

```
posição:             1    2    3    4    5    6    7    8    9    10
foto:                ·    ✓    ✓    ·    ·    ✓    ·    ·    ✓    ·
precisão em cada ✓:       1/2  2/3            3/6            4/9

AP = (0,50 + 0,67 + 0,50 + 0,44) ÷ 4 ≈ 0,53
```

Se todos os ✓ viessem primeiro, a AP seria 1,0.

**Resultado do BLR na validação**

| | AP |
|---|---|
| acaso (a fração de fotos com BLR) | 0,423 |
| heurística (Sprint 4) | 0,696 |
| modelo | 0,8192 |

**Como ler**
- O modelo quase dobra a AP do acaso e fica 0,12 acima da heurística.
- A AUC olha a lista inteira, e a AP olha o topo, onde está o alerta. Nos rótulos raros (BRT e DRK), a AP é a medida mais exigente.
- Cuidado: o s4e não calculou intervalo de confiança para a AP da heurística. A comparação rigorosa virá no bootstrap pareado.

Próximo: a AP dos outros 3 rótulos.

**Passo 9 — a AP dos 4 rótulos** *(código escrito por Claude, a pedido da Joyce)*

Um laço sobre os rótulos, como o do passo 7. Ele usa os scores que o passo 7 guardou em `scores_val` e calcula a AP de cada um na validação. O BLR volta a aparecer como conferência: ele tem de dar a mesma AP do passo 8.

In [ ]:
# Passo 9: a AP dos 4 rótulos na validação, com os scores que o passo 7 guardou em scores_val

# AP da heurística na validação (linha de base do Sprint 4, notebook baseline), para comparar
AP_HEURISTICA = {"BLR": 0.696, "BRT": 0.380, "DRK": 0.490, "FRM": 0.574}

linhas = []

for j, rotulo in enumerate(ROTULOS):
    # o gabarito deste rótulo na validação: a coluna j de y_val (como no passo 1)
    gabarito_val = y_val[:, j]

    # medir: a AP dos scores deste rótulo (scores_val é o dicionário do passo 7)
    ap = average_precision_score(gabarito_val, scores_val[rotulo])

    linhas.append(
        {
            "rótulo": rotulo,
            # a AP de quem ordena sem critério é a fração de fotos com o rótulo
            "AP acaso": gabarito_val.mean(),
            "AP heurística": AP_HEURISTICA[rotulo],
            "AP modelo": ap,
            "diferença": ap - AP_HEURISTICA[rotulo],
        }
    )

tabela_ap = pd.DataFrame(linhas)
print(tabela_ap.round(4).to_string(index=False))

# conferência: o BLR do laço tem de dar a mesma AP da célula do passo 8
assert abs(tabela_ap.loc[tabela_ap["rótulo"] == "BLR", "AP modelo"].iloc[0] - aps_blr) < 1e-12
print("conferência: o BLR do laço é igual ao do passo 8")

**Explicação — passo 9: a AP dos 4 rótulos**

O laço faz, para cada rótulo, o que o passo 8 fez para o BLR: pega o gabarito da validação e os scores que o passo 7 guardou, e calcula a AP.

**Resultado (validação, 7.750 fotos)**

| rótulo | AP do acaso | AP da heurística | AP do modelo | diferença |
|---|---|---|---|---|
| BLR | 0,4231 | 0,696 | 0,8192 | +0,1232 |
| BRT | 0,0614 | 0,380 | 0,4316 | +0,0516 |
| DRK | 0,0614 | 0,490 | 0,5386 | +0,0486 |
| FRM | 0,5610 | 0,574 | 0,7883 | +0,2143 |

**As duas métricas lado a lado**

| rótulo | AUC heurística | AUC modelo | AP heurística | AP modelo |
|---|---|---|---|---|
| BLR | 0,789 | 0,8455 | 0,696 | 0,8192 |
| BRT | 0,833 | 0,8450 | 0,380 | 0,4316 |
| DRK | 0,822 | 0,8755 | 0,490 | 0,5386 |
| FRM | 0,542 | 0,7687 | 0,574 | 0,7883 |

**Como ler**
- O modelo fica acima da heurística nos 4 rótulos, na AP e na AUC.
- **FRM** tem o maior ganho. A heurística (0,574) mal passava do acaso (0,561), e o modelo chega a 0,788.
- **BRT e DRK** ganham pouco em valor absoluto (cerca de +0,05), mas a referência deles é o acaso, que é só 0,06. Em relação a ele, o modelo faz 7 e 9 vezes mais (0,43 e 0,54). Uma AP baixa em valor absoluto é esperada em rótulos raros.
- No **BRT**, a AUC não mostrava ganho claro (0,845 dentro do intervalo da heurística). A AP aponta um ganho pequeno, de 0,38 para 0,43. Sem margem de erro, ainda não dá para afirmar.

**Cuidados**
1. Nenhuma das comparações tem intervalo de confiança do modelo. O próximo passo para isso é o bootstrap pareado.
2. As 248 fotos de validação com cópia no treino podem deixar as notas um pouco otimistas.
3. Modelos sem corte nem regularização: são os mais simples.

**Passo 10 — alinhar a heurística com a validação** 

O bootstrap pareado mede o modelo e a heurística nas mesmas fotos. Por isso o score da heurística de cada foto de validação entra aqui, na mesma ordem de `X_val`. O alinhamento é feito pelo nome do arquivo, e quatro conferências provam que ele está certo.

In [ ]:
# Passo 10: trazer os scores da heurística para a validação, na mesma ordem das fotos de X_val

# o CSV da heurística (as pontuações do guidance.ts em cada foto), o mesmo do notebook baseline.
# Antes vinha pela URL do GitHub, o que exigia internet. Agora vem do dataset joyceximenes/avia-guidance-scores,
# anexado a este notebook: é o mesmo arquivo (analysis/data/guidance_scores.csv, commit 9bc0e32), e o notebook
# roda sem internet
import glob

achados = glob.glob("/kaggle/input/**/guidance_scores.csv", recursive=True)
assert len(achados) == 1, f"esperava 1 guidance_scores.csv em /kaggle/input, achei: {achados}"
heuristica = pd.read_csv(achados[0])
print("CSV da heurística:", achados[0], heuristica.shape)


def chave_da_foto(caminho):
    # o nome do arquivo, sem pasta e sem extensão: é o que identifica a foto nas duas tabelas
    # (no manifesto: "val/VizWiz_val_00002854.png"; no CSV: "images/val/val/VizWiz_val_00002854.jpg")
    return caminho.str.split("/").str[-1].str.rsplit(".", n=1).str[0]


# as fotos de validação do manifesto, na ordem de X_val (a ordem que o resto do notebook usa)
manifest_val = manifest[manifest["split"] == "val"].reset_index(drop=True)
manifest_val["chave"] = chave_da_foto(manifest_val["arquivo"])

# as fotos de validação do CSV da heurística
heuristica["chave"] = chave_da_foto(heuristica["caminho"])
heuristica_val = heuristica[heuristica["split"] == "val"]

# conferência 1: as duas tabelas têm as mesmas fotos, e nenhum nome se repete
mesmas_fotos = (
    set(manifest_val["chave"]) == set(heuristica_val["chave"])
    and manifest_val["chave"].is_unique
    and heuristica_val["chave"].is_unique
)
print("tabelas com as mesmas fotos, sem nomes repetidos:", mesmas_fotos)
assert mesmas_fotos

# conferência 2: elas estão na mesma ordem? Se não estiverem, comparar por posição trocaria as fotos
mesma_ordem_antes = (manifest_val["chave"].to_numpy() == heuristica_val["chave"].to_numpy()).all()
print("antes de alinhar, na mesma ordem:", mesma_ordem_antes)

# alinhar: a heurística passa a seguir a ordem do manifesto, casando pelo nome da foto
heuristica_val = heuristica_val.set_index("chave").loc[manifest_val["chave"]].reset_index()

# conferência 3: agora a ordem é a mesma
mesma_ordem_depois = (manifest_val["chave"].to_numpy() == heuristica_val["chave"].to_numpy()).all()
print("depois de alinhar, na mesma ordem:", mesma_ordem_depois)
assert mesma_ordem_depois

# conferência 4: os rótulos da heurística são iguais aos de y_val (a prova de que a foto de cada linha é a mesma)
rotulos_iguais = (heuristica_val[ROTULOS].to_numpy() == y_val).all()
print("rótulos iguais aos de y_val:", rotulos_iguais)
assert rotulos_iguais

**Explicação — passo 10: alinhar a heurística com a validação**

O bootstrap pareado mede o modelo e a heurística **nas mesmas fotos**. Para isso, a linha *i* do score do modelo e a linha *i* do score da heurística têm de ser a mesma foto.

**A armadilha:** as duas tabelas têm as mesmas 7.750 fotos de validação, mas em **ordens diferentes** (a conferência 2 deu `False`).

```
manifesto (ordem de X_val):   foto A    foto B    foto C   ...
CSV da heurística:            foto C    foto A    foto B   ...      (desenho ilustrativo)

por posição:   score do modelo da foto A  ×  score da heurística da foto C    ✗
por nome:      score do modelo da foto A  ×  score da heurística da foto A    ✓
```

Comparar por posição misturaria fotos. Num teste por fora, isso fez a AUC da heurística no BLR cair de 0,789 para 0,58, quase acaso.

**O que a célula faz**
1. Lê o CSV da heurística pelo GitHub (por isso a Internet do notebook está ligada).
2. Cria uma **chave** em cada tabela: o nome do arquivo, sem pasta e sem extensão. É o que identifica a foto nas duas (`.png` num lado e `.jpg` no outro).
3. Reordena a heurística para seguir a ordem do manifesto, casando pela chave.

**As 4 conferências**

| # | o que confere | resultado |
|---|---|---|
| 1 | as duas tabelas têm as mesmas fotos, sem nomes repetidos | `True` |
| 2 | estão na mesma ordem **antes** de alinhar | `False` |
| 3 | estão na mesma ordem **depois** de alinhar | `True` |
| 4 | os rótulos da heurística são iguais aos de `y_val` | `True` |

A conferência 3 prova pelos nomes. A 4 confirma por outro caminho, com os rótulos de cada foto.

Próximo: os scores da heurística, com o sinal certo (mais alto = mais provável o problema).

**Passo 11 — os scores da heurística, com o sinal certo** *(código escrito por Claude, a pedido da Joyce)*

Para medir a heurística com AUC e AP, o score de cada foto precisa subir quando o problema é mais provável. Por isso `blurScore`, `brightnessScore` e `edgeScore` entram com sinal negativo (pouca nitidez, pouco brilho e poucas bordas indicam o problema), e `overexposedRatio` entra como está. É a mesma orientação do notebook `baseline`. A conferência é reproduzir as notas do Sprint 4.

In [ ]:
# Passo 11: os scores da heurística de cada foto de validação, com o sinal certo:
# quanto MAIOR o score, mais provável o problema (a mesma orientação do notebook baseline)
scores_heuristica = {
    "BLR": -heuristica_val["blurScore"].to_numpy(),  # pouca nitidez = desfocada
    "BRT": heuristica_val["overexposedRatio"].to_numpy(),  # mais pixels estourados = luz excessiva
    "DRK": -heuristica_val["brightnessScore"].to_numpy(),  # pouco brilho = escura
    "FRM": -heuristica_val["edgeScore"].to_numpy(),  # poucas bordas = mal enquadrada
}

# conferência: a AUC e a AP da heurística, calculadas aqui, têm de reproduzir as do Sprint 4 (linha de base)
linhas = []
for j, rotulo in enumerate(ROTULOS):
    auc = roc_auc_score(y_val[:, j], scores_heuristica[rotulo])
    ap = average_precision_score(y_val[:, j], scores_heuristica[rotulo])
    linhas.append({"rótulo": rotulo, "AUC aqui": auc, "AUC Sprint 4": AUC_HEURISTICA[rotulo], "AP aqui": ap, "AP Sprint 4": AP_HEURISTICA[rotulo]})
    assert abs(round(auc, 3) - AUC_HEURISTICA[rotulo]) < 1e-9, f"AUC da heurística não reproduz no {rotulo}"
    assert abs(round(ap, 3) - AP_HEURISTICA[rotulo]) < 1e-9, f"AP da heurística não reproduz no {rotulo}"

print(pd.DataFrame(linhas).round(4).to_string(index=False))
print("conferência: a heurística, alinhada pelo nome da foto, reproduz as notas do Sprint 4")

**Explicação — passo 11: os scores da heurística, com o sinal certo**

A AUC e a AP precisam de um score que **sobe** quando o problema é mais provável. A heurística era uma regra de corte (por exemplo, "`blurScore` < 120 = desfocada"), então em três casos o problema aparece nos valores **baixos**. Por isso esses entram com sinal negativo:

| rótulo | coluna | sinal | leitura |
|---|---|---|---|
| BLR | `blurScore` | − | pouca nitidez = desfocada |
| BRT | `overexposedRatio` | + | mais pixels estourados = luz excessiva |
| DRK | `brightnessScore` | − | pouco brilho = escura |
| FRM | `edgeScore` | − | poucas bordas = mal enquadrada |

**A conferência:** a AUC e a AP da heurística, calculadas aqui, reproduzem as notas do Sprint 4.

| rótulo | AUC aqui | AUC Sprint 4 | AP aqui | AP Sprint 4 |
|---|---|---|---|---|
| BLR | 0,7892 | 0,789 | 0,6962 | 0,696 |
| BRT | 0,8333 | 0,833 | 0,3802 | 0,380 |
| DRK | 0,8217 | 0,822 | 0,4904 | 0,490 |
| FRM | 0,5416 | 0,542 | 0,5737 | 0,574 |

Se o sinal ou o alinhamento estivessem errados, a conferência falharia. Alinhado por posição, a AUC do BLR cairia para 0,58.

**Passo 12 — uma reamostragem** *(código escrito por Claude, a pedido da Joyce)*

Sorteia 7.750 fotos com reposição e mede o modelo e a heurística nesse mesmo sorteio, só para o BLR e a AUC. É um único sorteio, para ver o que o bootstrap faz antes de repeti-lo.

In [ ]:
# Passo 12: UMA reamostragem do bootstrap, só para ver o que ele faz (rótulo BLR, métrica AUC)
rng = np.random.default_rng(42)  # semente fixa: o sorteio se repete igual
n_fotos = len(y_val)
j_blr = ROTULOS.index("BLR")

# sorteia, com reposição, n_fotos posições entre 0 e n_fotos - 1: uma foto pode sair várias vezes e outras nenhuma
sorteadas = rng.integers(0, n_fotos, size=n_fotos)

# o MESMO sorteio vale para o gabarito, para o score do modelo e para o da heurística: é isso que torna o bootstrap pareado
gabarito = y_val[sorteadas, j_blr]
auc_modelo = roc_auc_score(gabarito, scores_val["BLR"][sorteadas])
auc_heuristica = roc_auc_score(gabarito, scores_heuristica["BLR"][sorteadas])

print(f"fotos diferentes no sorteio: {len(np.unique(sorteadas))} de {n_fotos}")
print(f"AUC do modelo: {auc_modelo:.4f} | AUC da heurística: {auc_heuristica:.4f} | diferença: {auc_modelo - auc_heuristica:+.4f}")

**Explicação — passo 12: uma reamostragem**

```
7.750 fotos de validação
   │  sorteio com reposição: 7.750 posições (a mesma foto pode sair várias vezes)
   ▼
sorteadas ──▶ gabarito, score do modelo e score da heurística DESSAS fotos
   ▼
AUC do modelo 0,8434  |  AUC da heurística 0,7896  |  diferença +0,0538
```

O mesmo sorteio (`sorteadas`) vale para o gabarito, para o score do modelo e para o da heurística. É isso que torna o bootstrap **pareado**.

**O que se vê**
- O sorteio teve **4.854 fotos diferentes** das 7.750, ou seja, 62,6%. Cerca de 37% das fotos ficam de fora em cada reamostragem, e as outras entram repetidas. Pela teoria, o esperado é 63,2%.
- A diferença deste sorteio (+0,0538) é parecida com a medida nas 7.750 fotos (+0,0562), mas não igual. Essa variação de sorteio para sorteio é o que o bootstrap mede.
- A semente 42 faz o sorteio se repetir igual a cada execução.

**Passo 13 — 1.000 reamostragens do BLR** *(código escrito por Claude, a pedido da Joyce)*

Repete o sorteio do passo 12 mil vezes, com semente 42 (a mesma do intervalo de confiança do s4e). Cada sorteio dá uma diferença de AUC entre modelo e heurística, e o intervalo de confiança corta 2,5% de cada ponta dessas diferenças.

In [ ]:
# Passo 13: 1.000 reamostragens do BLR (AUC): a diferença modelo − heurística em cada uma
rng = np.random.default_rng(42)
n_boot = 1000

diferencas_blr = []
for _ in range(n_boot):
    sorteadas = rng.integers(0, n_fotos, size=n_fotos)
    gabarito = y_val[sorteadas, j_blr]
    auc_modelo = roc_auc_score(gabarito, scores_val["BLR"][sorteadas])
    auc_heuristica = roc_auc_score(gabarito, scores_heuristica["BLR"][sorteadas])
    diferencas_blr.append(auc_modelo - auc_heuristica)

# a diferença medida nas 7.750 fotos, sem sorteio, e o intervalo de confiança de 95%: corta 2,5% de cada ponta
medida_blr = roc_auc_score(y_val[:, j_blr], scores_val["BLR"]) - roc_auc_score(y_val[:, j_blr], scores_heuristica["BLR"])
ic_blr_auc = tuple(np.percentile(diferencas_blr, [2.5, 97.5]))

print(f"BLR, AUC: diferença medida {medida_blr:+.4f} | IC 95% [{ic_blr_auc[0]:+.4f}; {ic_blr_auc[1]:+.4f}]")
print("o intervalo inclui o 0?", ic_blr_auc[0] <= 0 <= ic_blr_auc[1])

**Explicação — passo 13: 1.000 reamostragens do BLR**

O sorteio do passo 12 é repetido 1.000 vezes. Cada um dá uma diferença de AUC entre modelo e heurística. O intervalo de confiança de 95% corta 2,5% de cada ponta dessas 1.000 diferenças.

```
|---------|---------|---------|---------|
0        0,02      0,04      0,06      0,08
                      [-----●----]
                    0,045  0,056  0,067
```

Diferença medida nas 7.750 fotos: **+0,0562** (o ●). Intervalo de 95%: **[+0,0452; +0,0670]**.

**Como ler**
- O intervalo fica **todo acima de 0**. Com 95% de confiança, o ganho do modelo na AUC do BLR está entre +0,045 e +0,067. Não é sorte desse conjunto de fotos.
- A semente 42 e as 1.000 reamostragens são as mesmas do intervalo de confiança do s4e, o que mantém a comparação consistente.

**Passo 14 — o bootstrap pareado dos 4 rótulos** *(código escrito por Claude, a pedido da Joyce)*

O mesmo procedimento do passo 13, agora para BLR, BRT, DRK e FRM, com AUC e AP. Em cada uma das 1.000 reamostragens, o mesmo sorteio de fotos serve para os 4 rótulos e para as duas métricas. O BLR/AUC refeito aqui tem de dar o mesmo intervalo do passo 13.

In [ ]:
# Passo 14: o bootstrap pareado dos 4 rótulos, com AUC e AP. Em cada reamostragem, o MESMO sorteio de fotos
# serve para os 4 rótulos e para as duas métricas
rng = np.random.default_rng(42)
metricas = {"AUC": roc_auc_score, "AP": average_precision_score}
diferencas = {(rotulo, nome): [] for rotulo in ROTULOS for nome in metricas}

for _ in range(n_boot):
    sorteadas = rng.integers(0, n_fotos, size=n_fotos)
    for j, rotulo in enumerate(ROTULOS):
        gabarito = y_val[sorteadas, j]
        for nome, metrica in metricas.items():
            modelo_no_sorteio = metrica(gabarito, scores_val[rotulo][sorteadas])
            heuristica_no_sorteio = metrica(gabarito, scores_heuristica[rotulo][sorteadas])
            diferencas[(rotulo, nome)].append(modelo_no_sorteio - heuristica_no_sorteio)

linhas = []
for j, rotulo in enumerate(ROTULOS):
    for nome, metrica in metricas.items():
        # a diferença medida nas 7.750 fotos, sem sorteio, e o intervalo de confiança de 95%
        medida = metrica(y_val[:, j], scores_val[rotulo]) - metrica(y_val[:, j], scores_heuristica[rotulo])
        baixo, alto = np.percentile(diferencas[(rotulo, nome)], [2.5, 97.5])
        linhas.append(
            {
                "rótulo": rotulo,
                "métrica": nome,
                "diferença": medida,
                "IC 2,5%": baixo,
                "IC 97,5%": alto,
                "inclui 0": bool(baixo <= 0 <= alto),
            }
        )

tabela_bootstrap = pd.DataFrame(linhas)
print(tabela_bootstrap.round(4).to_string(index=False))

# conferência: o BLR/AUC daqui tem de dar o mesmo intervalo do passo 13 (mesma semente, mesmos sorteios)
linha_blr = tabela_bootstrap[(tabela_bootstrap["rótulo"] == "BLR") & (tabela_bootstrap["métrica"] == "AUC")].iloc[0]
assert abs(linha_blr["IC 2,5%"] - ic_blr_auc[0]) < 1e-12 and abs(linha_blr["IC 97,5%"] - ic_blr_auc[1]) < 1e-12
print("conferência: o BLR/AUC é igual ao do passo 13")

**Explicação — passo 14: o bootstrap pareado dos 4 rótulos**

O mesmo procedimento do passo 13, para os 4 rótulos e as duas métricas. Em cada uma das 1.000 reamostragens, o mesmo sorteio de fotos serve para tudo. O BLR/AUC refeito aqui deu o mesmo intervalo do passo 13, o que confere o código.

**Diferença modelo − heurística, com intervalo de confiança de 95%**

| rótulo | métrica | diferença | IC 95% | o ganho é real? |
|---|---|---|---|---|
| BLR | AUC | +0,0562 | [+0,0452; +0,0670] | sim |
| BLR | AP | +0,1231 | [+0,1055; +0,1402] | sim |
| BRT | AUC | +0,0117 | [−0,0140; +0,0358] | **não dá para afirmar** (inclui 0) |
| BRT | AP | +0,0514 | [+0,0016; +0,0984] | por muito pouco (o limite inferior quase toca 0) |
| DRK | AUC | +0,0539 | [+0,0330; +0,0763] | sim |
| DRK | AP | +0,0482 | [+0,0124; +0,0843] | sim, mas o tamanho é incerto |
| FRM | AUC | +0,2271 | [+0,2105; +0,2414] | sim, e é grande |
| FRM | AP | +0,2147 | [+0,1985; +0,2297] | sim, e é grande |

**Como ler**
- Em 7 dos 8 intervalos, o ganho do modelo fica acima de 0. A exceção é a AUC do BRT, que inclui 0.
- **FRM:** o ganho é enorme e claro. A heurística estava no nível do acaso.
- **BLR:** o ganho é claro nas duas métricas.
- **DRK:** o ganho é real, mas o intervalo da AP é largo ([0,012; 0,084]), porque o rótulo é raro (6% das fotos).
- **BRT:** o ganho é pequeno e incerto. A AUC não distingue o modelo da heurística, e a AP passa de 0 por 0,0016.

**Para o artigo:** o modelo supera a heurística no BLR, no DRK e no FRM. No BRT não há evidência clara de ganho.

**Cuidados**
1. 248 fotos de validação (3,2%) têm cópia exata no treino, o que pode favorecer o modelo. A heurística não é treinada, então não é afetada. O efeito não foi medido.
2. O bootstrap mede só a variação de **sortear as fotos de validação**. Ele não inclui a variação de treinar o modelo de novo nem o ruído dos rótulos (kappa).
3. São 8 intervalos de 95%. Quanto mais comparações, maior a chance de uma delas errar por acaso.

**Passo 15 — previsões de validação cruzada no treino** *(código escrito por Claude, a pedido da Joyce)*

Escolher um corte exige scores de fotos que o modelo **não viu**, e a validação não pode ser usada para isso: ela só mede. A saída é dividir o treino em 5 pedaços. Cada pedaço é previsto por um modelo ajustado nos outros 4, e assim cada foto de treino ganha um score honesto, sem tocar na validação.

In [ ]:
# Passo 15: previsões de validação cruzada no TREINO, para escolher os cortes sem usar a validação
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

# a mesma receita (escala + regressão, max_iter=1000), mas dentro de um pipeline: a régua é medida de novo,
# só com as fotos de treino de cada rodada, para a foto "de fora" nunca influenciar a própria previsão
receita = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# 5 rodadas: o treino é dividido em 5 pedaços; cada pedaço é previsto por um modelo ajustado nos outros 4
divisao = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_cv = {}  # o score de cada foto de treino, vindo de um modelo que NÃO a viu
for j, rotulo in enumerate(ROTULOS):
    scores_cv[rotulo] = cross_val_predict(receita, X_treino, y_treino[:, j], cv=divisao, method="predict_proba")[:, 1]
    auc_cv = roc_auc_score(y_treino[:, j], scores_cv[rotulo])
    print(f"{rotulo}: AUC da validação cruzada no treino {auc_cv:.4f}")

**Explicação — passo 15: previsões de validação cruzada no treino**

**O problema.** Um corte sai de scores. Se os scores forem de fotos que o modelo usou para aprender, eles são otimistas, e o corte sai enviesado. A validação não pode ser usada para escolher, porque ela só mede. Faltam scores de fotos de treino que o modelo **não viu**.

**A solução: validação cruzada.** O treino é dividido em 5 pedaços, e cada pedaço é previsto por um modelo ajustado nos outros 4:

```
rodada 1:  [prever ] [treinar] [treinar] [treinar] [treinar]
rodada 2:  [treinar] [prever ] [treinar] [treinar] [treinar]
rodada 3:  [treinar] [treinar] [prever ] [treinar] [treinar]
rodada 4:  [treinar] [treinar] [treinar] [prever ] [treinar]
rodada 5:  [treinar] [treinar] [treinar] [treinar] [prever ]

cada foto de treino é prevista UMA vez, por um modelo que não a viu
```

**Detalhes do código**
- `StratifiedKFold`: cada pedaço mantém a mesma proporção de positivos. `shuffle=True` e semente 42 deixam a divisão repetível.
- O `make_pipeline(StandardScaler(), LogisticRegression(...))` faz a régua da escala ser medida **de novo, só nas fotos de treino de cada rodada**. Assim a foto "de fora" não influencia nem a régua.
- Os 5 modelos de cada rótulo servem **só para escolher o corte**. O modelo final continua sendo o do passo 7.

**Conferência de sanidade.** A AUC dessas previsões honestas no treino fica perto da AUC na validação:

| rótulo | AUC (validação cruzada, treino) | AUC (validação, passo 7) |
|---|---|---|
| BLR | 0,8446 | 0,8455 |
| BRT | 0,8376 | 0,8450 |
| DRK | 0,8663 | 0,8755 |
| FRM | 0,7745 | 0,7687 |

O treino honesto e a validação contam a mesma história. Isso dá confiança de que o corte escolhido no treino vale na validação.

**Passo 16 — os cortes** *(código escrito por Claude, a pedido da Joyce)*

Dois cortes, escolhidos só com os scores do passo 15: o de maior **F0,5** em cada rótulo (o critério do s5a, em que a precisão pesa mais que a revocação) e, no BLR, o corte da **precisão mínima de 80%** com a maior revocação possível, que é o teste que o s5a deixou para o modelo.

In [ ]:
# Passo 16: os cortes, escolhidos SÓ com as previsões de validação cruzada do treino
from sklearn.metrics import precision_recall_curve


def curva_do_treino(gabarito, scores):
    # precisão e revocação em cada corte possível; o último ponto da curva (precisão 1, revocação 0) não tem corte
    precisao, revocacao, cortes_possiveis = precision_recall_curve(gabarito, scores)
    return precisao[:-1], revocacao[:-1], cortes_possiveis


def corte_f05(gabarito, scores):
    # o corte de maior F0,5: equilibra precisão e revocação, com a precisão pesando mais (critério do s5a)
    precisao, revocacao, cortes_possiveis = curva_do_treino(gabarito, scores)
    f05 = np.divide(1.25 * precisao * revocacao, 0.25 * precisao + revocacao, out=np.zeros_like(precisao), where=(precisao + revocacao) > 0)
    melhor = np.argmax(f05)
    return cortes_possiveis[melhor], precisao[melhor], revocacao[melhor], f05[melhor]


def corte_com_precisao_minima(gabarito, scores, piso):
    # entre os cortes com precisão >= piso, o de maior revocação
    precisao, revocacao, cortes_possiveis = curva_do_treino(gabarito, scores)
    atendem = np.where(precisao >= piso)[0]
    assert len(atendem) > 0, f"nenhum corte chega a precisão {piso}"
    melhor = atendem[np.argmax(revocacao[atendem])]
    return cortes_possiveis[melhor], precisao[melhor], revocacao[melhor]


cortes = {}
linhas = []
for j, rotulo in enumerate(ROTULOS):
    corte, p, r, f = corte_f05(y_treino[:, j], scores_cv[rotulo])
    cortes[rotulo] = corte
    linhas.append({"rótulo": rotulo, "corte (F0,5)": corte, "precisão (treino)": p, "revocação (treino)": r, "F0,5 (treino)": f})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

# o teste que o s5a deixou para o modelo: no BLR, precisão mínima de 80% e, entre esses cortes, a maior revocação
corte_80, precisao_80, revocacao_80 = corte_com_precisao_minima(y_treino[:, 0], scores_cv["BLR"], 0.80)
print()
print(f"BLR, precisão mínima de 80%: corte {corte_80:.4f} | no treino: precisão {precisao_80:.4f}, revocação {revocacao_80:.4f}")

**Explicação — passo 16: os cortes**

Um corte é um valor de score: a foto recebe alerta quando o score é **maior ou igual** a ele. Dois jeitos de escolher, os dois só com as previsões do passo 15 (treino):

**1. Maior F0,5 (o critério do s5a).** Para cada corte possível, calcula-se precisão (P) e revocação (R), e o F0,5 junta as duas com a precisão pesando mais:

```
F0,5 = 1,25 × P × R ÷ (0,25 × P + R)
```

O corte escolhido é o de maior F0,5.

**2. Precisão mínima de 80% (o teste herdado do s5a, só no BLR).** Entre os cortes com P ≥ 0,80, fica o de maior revocação.

**Cortes escolhidos (no treino, com validação cruzada)**

| rótulo | corte (F0,5) | precisão | revocação | F0,5 |
|---|---|---|---|---|
| BLR | 0,6759 | 0,8280 | 0,5336 | 0,7457 |
| BRT | 0,5736 | 0,5256 | 0,2628 | 0,4380 |
| DRK | 0,8265 | 0,8145 | 0,3050 | 0,6105 |
| FRM | 0,5951 | 0,7631 | 0,6660 | 0,7415 |

Precisão mínima de 80% no BLR: **corte 0,6148**, com precisão 0,8000 e revocação 0,5818 no treino.

Nada da validação entrou nesta escolha.

**Passo 17 — medir os cortes na validação** *(código escrito por Claude, a pedido da Joyce)*

Aplica os cortes do passo 16 nos scores de validação do modelo ajustado com o treino inteiro (passo 7) e compara com a regra de alerta do app hoje (`guidance.ts`: BLR 120, DRK 30, BRT 0,22 e FRM fora da faixa 18–55). A regra do app, calculada aqui, precisa reproduzir as medidas do s5a e do Sprint 4.

In [ ]:
# Passo 17: medir na VALIDAÇÃO os cortes escolhidos no treino, e comparar com a regra do app hoje (guidance.ts)
from sklearn.metrics import fbeta_score, precision_score, recall_score

# a regra de alerta do app hoje, para cada rótulo (BRT só pelo overexposedRatio, como no "depois" do s5a)
regra_app = {
    "BLR": (heuristica_val["blurScore"] < 120).to_numpy(),
    "BRT": (heuristica_val["overexposedRatio"] > 0.22).to_numpy(),
    "DRK": (heuristica_val["brightnessScore"] < 30).to_numpy(),
    "FRM": ((heuristica_val["edgeScore"] > 55) | (heuristica_val["edgeScore"] < 18)).to_numpy(),
}


def medidas(gabarito, alerta):
    return precision_score(gabarito, alerta), recall_score(gabarito, alerta), fbeta_score(gabarito, alerta, beta=0.5)


# conferência: a regra do app, calculada aqui, reproduz a precisão e a revocação que o s5a e o Sprint 4 mediram
CONHECIDO = {"BLR": (0.7122, 0.5916), "BRT": (0.6364, 0.2059), "DRK": (0.8933, 0.2815), "FRM": (0.580, 0.745)}
for j, rotulo in enumerate(ROTULOS):
    p_regra, r_regra, _ = medidas(y_val[:, j], regra_app[rotulo])
    assert abs(p_regra - CONHECIDO[rotulo][0]) < 0.001 and abs(r_regra - CONHECIDO[rotulo][1]) < 0.001, f"a regra do app não reproduz no {rotulo}"
print("conferência: a regra do app reproduz a precisão e a revocação do s5a e do Sprint 4")
print()

linhas = []
for j, rotulo in enumerate(ROTULOS):
    p_regra, r_regra, f_regra = medidas(y_val[:, j], regra_app[rotulo])
    p_modelo, r_modelo, f_modelo = medidas(y_val[:, j], scores_val[rotulo] >= cortes[rotulo])
    linhas.append(
        {
            "rótulo": rotulo,
            "P regra": p_regra,
            "R regra": r_regra,
            "F0,5 regra": f_regra,
            "P modelo": p_modelo,
            "R modelo": r_modelo,
            "F0,5 modelo": f_modelo,
        }
    )
print("validação, corte do modelo escolhido pelo F0,5 no treino:")
print(pd.DataFrame(linhas).round(4).to_string(index=False))

# o teste dos 80% no BLR: o corte veio do treino; aqui só se mede
p80_val, r80_val, f80_val = medidas(y_val[:, 0], scores_val["BLR"] >= corte_80)
print()
print(f"BLR, corte da precisão mínima de 80% (escolhido no treino): validação precisão {p80_val:.4f}, revocação {r80_val:.4f}, F0,5 {f80_val:.4f}")

**Explicação — passo 17: medir os cortes na validação**

Os cortes do passo 16 são aplicados nos scores de validação (modelo ajustado com o treino inteiro, passo 7) e comparados com a **regra de alerta do app hoje**. A conferência confirmou que a regra, calculada aqui, reproduz a precisão e a revocação que o s5a e o Sprint 4 mediram.

**Validação, corte do modelo escolhido pelo F0,5 no treino**

| rótulo | P regra | R regra | F0,5 regra | P modelo | R modelo | F0,5 modelo |
|---|---|---|---|---|---|---|
| BLR | 0,7122 | 0,5916 | 0,6843 | 0,8543 | 0,5133 | 0,7541 |
| BRT | 0,6364 | 0,2059 | 0,4487 | 0,6789 | 0,2710 | 0,5218 |
| DRK | 0,8933 | 0,2815 | 0,6227 | 0,8767 | 0,2689 | 0,6038 |
| FRM | 0,5797 | 0,7447 | 0,6065 | 0,7532 | 0,6746 | 0,7360 |

**O teste dos 80% no BLR (passou)**

O corte da precisão mínima de 80%, escolhido no treino, deu na validação **precisão 0,8272 e revocação 0,5621** (F0,5 0,7559). A regra do app tem 0,7122 e 0,5916. Em palavras: **+11,5 pontos de precisão por −3,0 de revocação**. No s5a, o melhor que o `blurScore` fazia era chegar a ~76% de precisão no treino, com revocação de ~21%. O modelo chega a mais de 80% de precisão com revocação de 56%, perto da regra de hoje.

**Como ler**
- **BLR e FRM:** a precisão sobe muito (+14 e +17 pontos), e a revocação cai um pouco (−8 e −7 pontos).
- **BRT:** precisão e revocação sobem juntas (63,6% → 67,9% e 20,6% → 27,1%).
- **DRK:** o modelo **não supera a regra**. O F0,5 é parecido, um pouco menor (0,604 contra 0,623). O ganho de AUC e AP do DRK não virou ganho no ponto de operação.

**Passo 18 — margem de erro dos cortes** *(código escrito por Claude, a pedido da Joyce)*

O mesmo bootstrap pareado do passo 14 (1.000 reamostragens, semente 42), agora para precisão, revocação e F0,5 no corte escolhido. Os cortes ficam **fixos**, escolhidos no treino, e só as fotos de validação são sorteadas. O sorteio é o mesmo para o modelo e para a regra do app.

In [ ]:
# Passo 18: margem de erro (bootstrap pareado: 1.000 reamostragens, semente 42) das medidas no corte escolhido


def medidas_rapidas(gabarito, alerta):
    # precisão, revocação e F0,5 a partir das contagens: o mesmo resultado do sklearn, bem mais rápido
    acertos = np.sum(gabarito & alerta)
    falsos_alarmes = np.sum(~gabarito & alerta)
    perdidos = np.sum(gabarito & ~alerta)
    precisao = acertos / (acertos + falsos_alarmes) if acertos + falsos_alarmes else 0.0
    revocacao = acertos / (acertos + perdidos) if acertos + perdidos else 0.0
    f05 = 1.25 * precisao * revocacao / (0.25 * precisao + revocacao) if precisao + revocacao else 0.0
    return np.array([precisao, revocacao, f05])


# os casos comparados: cada um tem o gabarito, o alerta do modelo (no corte do treino) e o alerta da regra do app
casos = []
for j, rotulo in enumerate(ROTULOS):
    casos.append((f"{rotulo} (corte F0,5)", y_val[:, j].astype(bool), scores_val[rotulo] >= cortes[rotulo], regra_app[rotulo]))
casos.append(("BLR (precisão mín. 80%)", y_val[:, 0].astype(bool), scores_val["BLR"] >= corte_80, regra_app["BLR"]))

# conferência: a versão rápida dá o mesmo que o sklearn do passo 17
for nome, gabarito, alerta_modelo, alerta_regra in casos:
    esperado = np.array(medidas(gabarito, alerta_modelo))
    assert np.allclose(medidas_rapidas(gabarito, alerta_modelo), esperado), f"medidas rápidas diferentes no caso {nome}"
print("conferência: as medidas rápidas são iguais às do sklearn")

rng = np.random.default_rng(42)
modelo_nos_sorteios = {nome: [] for nome, *_ in casos}
regra_nos_sorteios = {nome: [] for nome, *_ in casos}
for _ in range(n_boot):
    sorteadas = rng.integers(0, n_fotos, size=n_fotos)  # o mesmo sorteio para todos os casos, para o modelo e para a regra
    for nome, gabarito, alerta_modelo, alerta_regra in casos:
        modelo_nos_sorteios[nome].append(medidas_rapidas(gabarito[sorteadas], alerta_modelo[sorteadas]))
        regra_nos_sorteios[nome].append(medidas_rapidas(gabarito[sorteadas], alerta_regra[sorteadas]))

linhas = []
for nome, gabarito, alerta_modelo, alerta_regra in casos:
    medido_modelo = medidas_rapidas(gabarito, alerta_modelo)
    medido_regra = medidas_rapidas(gabarito, alerta_regra)
    sorteios_modelo = np.array(modelo_nos_sorteios[nome])
    sorteios_dif = sorteios_modelo - np.array(regra_nos_sorteios[nome])
    for k, medida in enumerate(["precisão", "revocação", "F0,5"]):
        ic_modelo = np.percentile(sorteios_modelo[:, k], [2.5, 97.5])
        ic_dif = np.percentile(sorteios_dif[:, k], [2.5, 97.5])
        linhas.append(
            {
                "caso": nome,
                "medida": medida,
                "modelo": medido_modelo[k],
                "IC modelo": f"[{ic_modelo[0]:.3f}; {ic_modelo[1]:.3f}]",
                "regra do app": medido_regra[k],
                "diferença": medido_modelo[k] - medido_regra[k],
                "IC diferença": f"[{ic_dif[0]:+.3f}; {ic_dif[1]:+.3f}]",
                "inclui 0": bool(ic_dif[0] <= 0 <= ic_dif[1]),
            }
        )

tabela_cortes = pd.DataFrame(linhas)
print(tabela_cortes.round(4).to_string(index=False))

**Explicação — passo 18: margem de erro dos cortes**

O bootstrap pareado do passo 14, agora para precisão, revocação e F0,5 no corte escolhido. Os cortes ficam **fixos** (vieram do treino), e só as fotos de validação são sorteadas, as mesmas para o modelo e para a regra do app. O código confere as medidas rápidas com as do sklearn.

**Diferença modelo − regra do app, com intervalo de confiança de 95%**

| caso | precisão | revocação | F0,5 |
|---|---|---|---|
| BLR (corte F0,5) | +0,142 [+0,124; +0,161] | −0,078 [−0,097; −0,058] | +0,070 [+0,054; +0,087] |
| BLR (precisão mín. 80%) | +0,115 [+0,097; +0,133] | −0,030 [−0,047; −0,010] | +0,072 [+0,056; +0,088] |
| BRT | +0,043 [−0,043; +0,128] | +0,065 [+0,024; +0,107] | +0,073 [+0,009; +0,137] |
| DRK | −0,017 [−0,069; +0,038] | −0,013 [−0,047; +0,022] | −0,019 [−0,065; +0,024] |
| FRM | +0,174 [+0,162; +0,185] | −0,070 [−0,088; −0,053] | +0,130 [+0,117; +0,141] |

**Como ler**
- **Teste dos 80% no BLR:** a precisão do modelo na validação, 0,827, tem intervalo de **[0,810; 0,843]**. O limite inferior fica acima de 0,80, então o teste passa com margem, e o ganho de precisão sobre a regra do app é real (+0,115 [+0,097; +0,133]). A revocação cai pouco (−0,030 [−0,047; −0,010]).
- **BLR e FRM (corte F0,5):** ganho real de precisão e de F0,5, pagando alguns pontos de revocação.
- **BRT:** o F0,5 melhora (+0,073), mas o limite inferior é só +0,009. A revocação sobe de verdade, e a precisão não se distingue da regra (o intervalo inclui 0). O ganho é real, porém pequeno e incerto.
- **DRK:** todos os intervalos incluem 0, e as estimativas são levemente negativas. **Não há evidência de que o modelo seja melhor, nem pior, que a regra no ponto de operação.**

**Cuidados**
1. O bootstrap mede só o sorteio das fotos de validação. Não inclui a variação de treinar o modelo de novo nem o ruído dos rótulos.
2. Os cortes vieram de modelos treinados com 80% do treino. A validação mede o modelo com o treino inteiro. O resultado no BLR (precisão 0,827 contra o piso de 0,80) mostra que isso não deixou o corte otimista.
3. 15 intervalos de 95% ao mesmo tempo.

**Passo 19 — relato das duplicatas** *(código escrito por Claude, a pedido da Joyce)*

Decisão da Joyce: as duplicatas só são **relatadas**. Esta célula conta quantas fotos têm cópia exata e quantas atravessam treino e validação. Nenhuma foto foi removida e nenhuma análise anterior foi alterada.

In [ ]:
# Passo 19: relato das duplicatas. Só contagem: nenhuma foto foi removida e nenhuma análise foi alterada
import hashlib

# duas fotos são cópias quando as 960 colunas são idênticas (conferido por fora: as imagens 224 x 224 também são idênticas)
impressao = [hashlib.md5(linha.tobytes()).hexdigest() for linha in X]
grupo = pd.factorize(pd.Series(impressao))[0]  # um número de grupo para cada impressão diferente

tabela_dup = pd.DataFrame({"grupo": grupo, "split": manifest["split"].to_numpy()})
tamanho_do_grupo = tabela_dup.groupby("grupo").size()
grupos_com_copia = tamanho_do_grupo[tamanho_do_grupo > 1].index

print(f"grupos com mais de uma foto: {len(grupos_com_copia)} | fotos nesses grupos: {int(tamanho_do_grupo[grupos_com_copia].sum())}")

# grupos que têm foto de treino e de validação ao mesmo tempo
splits_por_grupo = tabela_dup[tabela_dup["grupo"].isin(grupos_com_copia)].groupby("grupo")["split"].nunique()
print(f"grupos que atravessam treino e validação: {int((splits_por_grupo > 1).sum())}")

# fotos de validação que têm cópia exata no treino
grupos_do_treino = set(tabela_dup.loc[tabela_dup["split"] == "train", "grupo"])
com_copia_no_treino = int(tabela_dup.loc[tabela_dup["split"] == "val", "grupo"].isin(grupos_do_treino).sum())
print(f"fotos de validação com cópia no treino: {com_copia_no_treino} de {len(y_val)} ({com_copia_no_treino / len(y_val):.1%})")

# cópias com rótulos diferentes: os anotadores discordaram sobre a mesma imagem
combinacao_de_rotulos = manifest[ROTULOS].astype(str).agg("".join, axis=1)
rotulos_por_grupo = pd.DataFrame({"grupo": grupo, "rotulos": combinacao_de_rotulos}).groupby("grupo")["rotulos"].nunique()
divergentes = int((rotulos_por_grupo[grupos_com_copia] > 1).sum())
print(f"grupos com cópias de rótulos diferentes: {divergentes} de {len(grupos_com_copia)} ({divergentes / len(grupos_com_copia):.0%})")

**Explicação — passo 19: relato das duplicatas**

Por decisão da Joyce, as duplicatas só são **relatadas**: nenhuma foto foi removida, e nenhuma análise anterior foi alterada. Duas fotos contam como cópias quando as 960 colunas são idênticas (por fora, conferiu-se que as imagens de 224 × 224 também são idênticas pixel a pixel).

| o que | quantas |
|---|---|
| grupos com mais de uma foto | 673 (1.477 fotos no total) |
| grupos que têm foto de treino e de validação | 236 |
| fotos de validação com cópia exata no treino | **248 de 7.750 (3,2%)** |
| grupos cujas cópias têm rótulos diferentes | **294 de 673 (44%)** |

**O que isso pode significar (hipóteses, não medidas)**
- **Vazamento:** as 248 fotos de validação com cópia no treino podem receber notas um pouco melhores que fotos realmente novas, porque o modelo viu a imagem idêntica. A regra do app não é treinada, então não é afetada. As comparações do modelo com a heurística podem estar levemente favoráveis ao modelo.
- **Ruído de rótulo:** em 44% dos grupos, os anotadores discordaram sobre a mesma imagem. Isso mede o ruído dos rótulos direto e combina com o kappa de Fleiss (0,42 no BLR).
- **Validação cruzada do passo 15:** cópias de uma mesma foto podem cair em pedaços diferentes. Num teste feito fora do notebook, com a mesma versão do scikit-learn nas duas pontas, refazer a validação cruzada mantendo as cópias no mesmo pedaço não mudou o resultado do BLR no corte de 80%: precisão 0,828 e revocação 0,560, contra 0,827 e 0,563 com a divisão comum.

**O que não foi feito:** medir quanto as métricas mudariam sem essas 248 fotos. O efeito continua **não medido**.

## 5. Comparação e análise de erro (s6a)

Heurística calibrada (a regra do app hoje, `guidance.ts`) × modelo do s5c, nas mesmas 7.750 fotos de validação. Critério da Joyce: **menos alarmes falsos**, porque eles frustram a experiência; o app é tratado como prova de conceito (a conclusão é uma recomendação, sem integrar o modelo). A causa dos erros fica para trabalho futuro.


contagem de falsos alarmes para BLR

In [ ]:
fp_blr = (regra_app["BLR"] == True) & (blr_val == 0)
print(f"falsos positivos: {fp_blr.sum()}")

fn_blr = (regra_app["BLR"] == False) & (blr_val == 1)
print(f"não detectados (falsos negativos): {fn_blr.sum()}")

**Contagem dos erros do modelo no BLR** *(código escrito por Claude, a pedido da Joyce)*


In [ ]:
# o alerta do modelo: o score passa do corte escolhido no treino pelo F0,5 (passo 16)
alerta_modelo_blr = scores_val["BLR"] >= cortes["BLR"]

# a mesma conta da célula de cima; só muda de onde vem o alerta
fp_modelo_blr = (alerta_modelo_blr == True) & (blr_val == 0)  # alertou, mas não tinha BLR: falso alarme
fn_modelo_blr = (alerta_modelo_blr == False) & (blr_val == 1)  # tinha BLR, mas não alertou: não detectado

print(f"modelo     | BLR | falsos alarmes (FP): {fp_modelo_blr.sum()} | não detectados (FN): {fn_modelo_blr.sum()}")
print(f"heurística | BLR | falsos alarmes (FP): {fp_blr.sum()} | não detectados (FN): {fn_blr.sum()}")


**Passo 20 — os erros dos 4 rótulos, contados em fotos** *(código escrito por Claude, a pedido da Joyce)*


In [ ]:
# Passo 20: os erros de cada abordagem, contados em fotos, para os 4 rótulos (validação, no corte de cada uma)

# o alerta de cada abordagem, por rótulo: True = avisou
alertas_por_abordagem = {
    "heurística": regra_app,  # a regra do app hoje (passo 17)
    "modelo": {rotulo: scores_val[rotulo] >= cortes[rotulo] for rotulo in ROTULOS},  # corte do F0,5 no treino (passo 16)
}

linhas = []
for j, rotulo in enumerate(ROTULOS):
    gabarito = y_val[:, j] == 1  # True = a foto tem o problema
    for abordagem, alertas in alertas_por_abordagem.items():
        alerta = np.asarray(alertas[rotulo], dtype=bool)
        acertos = int(np.sum(alerta & gabarito))  # avisou e tinha o problema
        fp = int(np.sum(alerta & ~gabarito))  # avisou, mas não tinha: falso alarme
        fn = int(np.sum(~alerta & gabarito))  # tinha, mas não avisou: não detectado
        linhas.append(
            {
                "rótulo": rotulo,
                "abordagem": abordagem,
                "fotos com o problema": int(gabarito.sum()),
                "alertas": int(alerta.sum()),
                "acertos": acertos,
                "falsos alarmes (FP)": fp,
                "não detectados (FN)": fn,
                "FP a cada 100 alertas": 100 * fp / alerta.sum(),
                "FN a cada 100 problemas": 100 * fn / gabarito.sum(),
            }
        )

tabela_erros = pd.DataFrame(linhas)
print(tabela_erros.round(1).to_string(index=False))

# o que muda do modelo para a heurística, em fotos
por_caso = tabela_erros.set_index(["rótulo", "abordagem"])
print()
for rotulo in ROTULOS:
    h, m = por_caso.loc[(rotulo, "heurística")], por_caso.loc[(rotulo, "modelo")]
    fp_h, fp_m = int(h["falsos alarmes (FP)"]), int(m["falsos alarmes (FP)"])
    fn_h, fn_m = int(h["não detectados (FN)"]), int(m["não detectados (FN)"])
    print(f"{rotulo}: falsos alarmes {fp_h} -> {fp_m} ({fp_m - fp_h:+d}) | não detectados {fn_h} -> {fn_m} ({fn_m - fn_h:+d})")

# conferência 1: o BLR da tabela é igual às contagens feitas nas células de cima
assert por_caso.loc[("BLR", "heurística"), "falsos alarmes (FP)"] == fp_blr.sum()
assert por_caso.loc[("BLR", "heurística"), "não detectados (FN)"] == fn_blr.sum()
assert por_caso.loc[("BLR", "modelo"), "falsos alarmes (FP)"] == fp_modelo_blr.sum()
assert por_caso.loc[("BLR", "modelo"), "não detectados (FN)"] == fn_modelo_blr.sum()

# conferência 2: precisão e revocação tiradas das contagens são as mesmas dos passos 17 e 18
for j, rotulo in enumerate(ROTULOS):
    for abordagem, alertas in alertas_por_abordagem.items():
        linha = por_caso.loc[(rotulo, abordagem)]
        esperado = medidas_rapidas(y_val[:, j].astype(bool), np.asarray(alertas[rotulo], dtype=bool))
        assert abs(linha["acertos"] / linha["alertas"] - esperado[0]) < 1e-12, (rotulo, abordagem, "precisão")
        assert abs(linha["acertos"] / linha["fotos com o problema"] - esperado[1]) < 1e-12, (rotulo, abordagem, "revocação")
print()
print("conferências: o BLR bate com as células de cima; precisão e revocação batem com os passos 17 e 18")


**Explicação — passo 20: os erros contados em fotos**

Cada foto tem duas informações: o **gabarito** (tem o problema?) e o **alerta** (a abordagem avisou?). Cada tipo de erro é uma combinação das duas:

```
falso alarme  (FP)  = avisou       E  não tinha o problema
não detectado (FN)  = não avisou   E  tinha o problema
```

A Joyce escreveu essa conta para a heurística no BLR (784 falsos alarmes, 1.339 não detectados); a célula seguinte fez o mesmo para o modelo no BLR (287 e 1.596). O passo 20 repete a conta para os 4 rótulos e confere que o BLR bate com essas células e que a precisão e a revocação tiradas das contagens são as mesmas dos passos 17 e 18.

**O que muda da heurística para o modelo, em fotos (validação, 7.750 fotos):**

- **BLR:** falsos alarmes **784 → 287** (de 29 para 15 a cada 100 alertas); não detectados 1.339 → 1.596 (+257).
- **FRM:** falsos alarmes **2.348 → 961** (de 42 para 25 a cada 100 alertas); não detectados 1.110 → 1.415 (+305).
- **DRK:** praticamente igual: falsos alarmes 16 → 18, não detectados 342 → 348.
- **BRT:** falsos alarmes 56 → 61; não detectados 378 → 347 (−31).

**Leitura:** no BLR e no FRM, o modelo troca muitos alarmes falsos por alguns problemas a mais não detectados. É o efeito do corte pelo F0,5, que dá mais peso à precisão, o critério da Joyce. Para saber se a diferença é real, e não sorte do sorteio das fotos, vale o intervalo do passo 18: no BLR e no FRM, os intervalos da precisão e da revocação não incluem o 0. No DRK, todos incluem; no BRT, a precisão inclui.


**Passo 21 — as figuras do capítulo de resultados** *(código escrito por Claude, a pedido da Joyce)*


In [ ]:
# Passo 21: as figuras do capítulo de resultados, gravadas em /kaggle/working/figuras
import os

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import roc_curve

PASTA_FIG = "/kaggle/working/figuras"
os.makedirs(PASTA_FIG, exist_ok=True)

COR = {"modelo": "#2a78d6", "heurística": "#eb6834"}  # azul × laranja: par conferido para daltonismo
TRACO = {"modelo": "-", "heurística": "--"}  # o traço também distingue as duas, sem depender só da cor
MARCA = {"modelo": "o", "heurística": "s"}
NOME = {"BLR": "BLR (desfoque)", "BRT": "BRT (claro demais)", "DRK": "DRK (escuro)", "FRM": "FRM (enquadramento)"}
TINTA, TINTA_2, GRADE, EIXO = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
SCORES = {"heurística": scores_heuristica, "modelo": scores_val}  # scores com o sinal certo (passos 7 e 11)

plt.rcParams.update(
    {
        "font.size": 10,
        "axes.edgecolor": EIXO,
        "axes.labelcolor": TINTA_2,
        "xtick.color": TINTA_2,
        "ytick.color": TINTA_2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
    }
)


def virgula(numero, casas):
    # números com vírgula decimal, como no texto da monografia
    return f"{numero:.{casas}f}".replace(".", ",")


EIXO_COM_VIRGULA = FuncFormatter(lambda valor, _: virgula(valor, 1))


def moldura(ax, titulo, nome_x, nome_y):
    ax.xaxis.set_major_formatter(EIXO_COM_VIRGULA)
    ax.yaxis.set_major_formatter(EIXO_COM_VIRGULA)
    ax.set_title(titulo, loc="left", color=TINTA, fontsize=11)
    ax.set_xlabel(nome_x)
    ax.set_ylabel(nome_y)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal")
    ax.grid(color=GRADE, linewidth=0.6)
    ax.set_axisbelow(True)


def ponto_de_operacao(j, rotulo, abordagem):
    # precisão, revocação e taxa de falsos alarmes no corte que cada abordagem usa de fato
    gabarito = y_val[:, j].astype(bool)
    alerta = np.asarray(alertas_por_abordagem[abordagem][rotulo], dtype=bool)
    precisao, revocacao, _ = medidas_rapidas(gabarito, alerta)
    taxa_fp = np.sum(alerta & ~gabarito) / np.sum(~gabarito)
    return precisao, revocacao, taxa_fp


NOTA_PONTOS = (
    "Marcadores: o ponto em que cada abordagem opera (o corte usado). No FRM, a regra do app usa dois cortes "
    "no edgeScore (acima de 55 ou abaixo de 18), por isso o ponto da heurística fica fora da curva dela. No DRK, "
    "os dois pontos quase coincidem."
)

# Figura 1: curvas ROC
fig, eixos = plt.subplots(2, 2, figsize=(8.5, 9))
for ax, (j, rotulo) in zip(eixos.flat, enumerate(ROTULOS)):
    gabarito = y_val[:, j]
    ax.plot([0, 1], [0, 1], color=EIXO, linewidth=1, linestyle=":")
    for abordagem in ("heurística", "modelo"):
        taxa_fp_curva, revocacao_curva, _ = roc_curve(gabarito, SCORES[abordagem][rotulo])
        auc = roc_auc_score(gabarito, SCORES[abordagem][rotulo])
        ax.plot(taxa_fp_curva, revocacao_curva, color=COR[abordagem], linestyle=TRACO[abordagem], linewidth=2,
                label=f"{abordagem} (AUC {virgula(auc, 3)})")
        _, r, t = ponto_de_operacao(j, rotulo, abordagem)
        ax.plot(t, r, marker=MARCA[abordagem], markersize=8, color=COR[abordagem], markeredgecolor="white",
                markeredgewidth=1.5, linestyle="none", zorder=5)
    moldura(ax, NOME[rotulo], "taxa de falsos alarmes", "revocação")
    ax.legend(loc="lower right", frameon=False, fontsize=9)
fig.suptitle("Curvas ROC na validação (7.750 fotos): heurística × modelo", x=0.02, ha="left", color=TINTA, fontsize=12)
fig.text(0.02, 0.005, NOTA_PONTOS, fontsize=8, color=TINTA_2, wrap=True)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
fig.savefig(f"{PASTA_FIG}/s6a_curvas_roc.png")
plt.show()

# Figura 2: curvas de precisão-revocação
fig, eixos = plt.subplots(2, 2, figsize=(8.5, 9))
for ax, (j, rotulo) in zip(eixos.flat, enumerate(ROTULOS)):
    gabarito = y_val[:, j]
    acaso = gabarito.mean()
    ax.axhline(acaso, color=EIXO, linewidth=1, linestyle=":")
    # o rótulo do acaso fica embaixo da linha, no meio: é a região vazia nos 4 gráficos
    ax.text(0.5, acaso - 0.015, f"acaso ({virgula(acaso, 2)})", ha="center", va="top", fontsize=8, color=TINTA_2)
    for abordagem in ("heurística", "modelo"):
        precisao_curva, revocacao_curva, _ = precision_recall_curve(gabarito, SCORES[abordagem][rotulo])
        ap = average_precision_score(gabarito, SCORES[abordagem][rotulo])
        ax.plot(revocacao_curva, precisao_curva, color=COR[abordagem], linestyle=TRACO[abordagem], linewidth=2,
                label=f"{abordagem} (AP {virgula(ap, 3)})")
        p, r, _ = ponto_de_operacao(j, rotulo, abordagem)
        ax.plot(r, p, marker=MARCA[abordagem], markersize=8, color=COR[abordagem], markeredgecolor="white",
                markeredgewidth=1.5, linestyle="none", zorder=5)
    moldura(ax, NOME[rotulo], "revocação", "precisão")
    ax.legend(loc="upper right", frameon=False, fontsize=9)
fig.suptitle("Curvas de precisão-revocação na validação: heurística × modelo", x=0.02, ha="left", color=TINTA, fontsize=12)
fig.text(0.02, 0.005, NOTA_PONTOS, fontsize=8, color=TINTA_2, wrap=True)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
fig.savefig(f"{PASTA_FIG}/s6a_curvas_precisao_revocacao.png")
plt.show()

# Figura 3: os dois tipos de erro, em barras (o critério da Joyce: alarmes falsos)
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
posicoes = np.arange(len(ROTULOS))
altura = 0.38
paineis = [
    ("FP a cada 100 alertas", "Alarmes falsos a cada 100 alertas"),
    ("FN a cada 100 problemas", "Não detectados a cada 100 fotos com o problema"),
]
for ax, (coluna, titulo) in zip(eixos, paineis):
    for k, abordagem in enumerate(("heurística", "modelo")):
        valores = [por_caso.loc[(rotulo, abordagem), coluna] for rotulo in ROTULOS]
        y = posicoes + (k - 0.5) * (altura + 0.03)  # 0,03 = folga entre as duas barras de cada rótulo
        ax.barh(y, valores, height=altura, color=COR[abordagem], label=abordagem, linewidth=0,
                hatch="///" if abordagem == "heurística" else None, edgecolor="white")
        for yi, v in zip(y, valores):
            ax.text(v + 1.2, yi, f"{v:.0f}", va="center", fontsize=9, color=TINTA)
    ax.set_xlim(0, 100)
    ax.set_title(titulo, loc="left", color=TINTA, fontsize=11)
    ax.grid(axis="x", color=GRADE, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
eixos[0].set_yticks(posicoes, [NOME[r] for r in ROTULOS])
eixos[0].invert_yaxis()  # BLR em cima; o eixo é compartilhado, então inverte uma vez só
eixos[0].legend(loc="lower right", frameon=False)
fig.suptitle("Os dois tipos de erro na validação, no corte de cada abordagem", x=0.02, ha="left", color=TINTA, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f"{PASTA_FIG}/s6a_erros_por_tipo.png")
plt.show()

print("figuras gravadas:", sorted(os.listdir(PASTA_FIG)))


**Explicação — passo 21: as figuras**

As três figuras ficam em `/kaggle/working/figuras` (saída do notebook) e vão para o capítulo de resultados. Em todas, **azul contínuo = modelo** e **laranja tracejado = heurística**; o traço diferente garante a leitura em preto e branco e para quem não distingue cores.

- **`s6a_curvas_roc.png`:** revocação × taxa de falsos alarmes em todos os cortes possíveis. Quanto mais a curva sobe para o canto de cima à esquerda, melhor a ordenação (AUC na legenda).
- **`s6a_curvas_precisao_revocacao.png`:** precisão × revocação em todos os cortes. É a figura que mais conversa com o critério da Joyce, porque a precisão é "quantos dos alertas eram de verdade" (AP na legenda; a linha pontilhada é o acaso).
- **`s6a_erros_por_tipo.png`:** os dois tipos de erro em barras, no corte que cada abordagem usa de fato.

Os **marcadores** (círculo = modelo, quadrado = heurística) mostram o ponto em que cada abordagem opera.

**O que as curvas de precisão-revocação acrescentam:**

- **BLR e FRM:** a curva do modelo fica **acima da heurística em todas as revocações**. O modelo não dá menos alarmes falsos só porque avisa menos: para encontrar a mesma quantidade de problemas, ele sempre erra menos. No FRM, a heurística fica quase em cima da linha do acaso.
- **DRK:** as curvas se cruzam. Na região das revocações baixas, onde as duas abordagens operam (revocação perto de 0,27), a heurística é tão precisa quanto o modelo, e às vezes mais. Isso explica por que o ganho de AUC e AP do DRK não vira ganho no corte.
- **BRT:** as curvas também se cruzam nas revocações baixas (abaixo de ~0,15 a heurística chega a ficar acima); o modelo passa à frente perto de 0,2 e só se afasta de verdade acima de ~0,3 de revocação.

**Cuidado de leitura:** no FRM, a regra do app usa dois cortes no `edgeScore` (acima de 55 ou abaixo de 18), por isso o ponto da heurística fica fora da curva dela, que usa um corte só.


## Conclusão do s6a — qual abordagem atende melhor o uso real *(texto escrito por Claude, a pedido da Joyce)*

**Critério (Joyce):** menos alarmes falsos, porque eles frustram a experiência. O app é uma prova de conceito; a conclusão é uma recomendação, sem integrar o modelo.

**Recomendação: o modelo, para BLR e FRM.** Na validação, no corte escolhido no treino:

- **BLR:** falsos alarmes de 784 para 287 (de 29 para 15 a cada 100 alertas); precisão +0,142 [+0,124; +0,161].
- **FRM:** falsos alarmes de 2.348 para 961 (de 42 para 25 a cada 100 alertas); precisão +0,174 [+0,162; +0,185].
- Nos dois, a curva de precisão-revocação do modelo fica acima da heurística em todas as revocações; o ganho não depende do corte escolhido.
- **O custo:** o modelo deixa passar mais problemas: revocação −0,078 [−0,097; −0,058] no BLR (+257 fotos não detectadas) e −0,070 [−0,088; −0,053] no FRM (+305). É a troca que o F0,5 escolhe de propósito, e a monografia precisa declará-la.

**DRK: sem diferença no uso.** O modelo ordena melhor (AUC +0,054 e AP +0,048, intervalos sem o 0), mas no corte as duas abordagens quase coincidem (falsos alarmes 16 × 18; não detectados 342 × 348; todos os intervalos incluem o 0). A regra do app já é precisa (89%).

**BRT: sem evidência de ganho em alarmes falsos.** A precisão não se distingue (+0,043 [−0,043; +0,128]); o modelo encontra um pouco mais (revocação +0,065 [+0,024; +0,107], 31 fotos), e a AUC não se distingue da heurística.

**Limites (trabalho futuro):**

- Prova de conceito: as fotos do VizWiz são finais, e o app analisa quadros de vídeo ao vivo. A comparação é um indício para o uso real, não uma medida dele.
- O modelo não roda no app; integrá-lo exigiria decidir onde ele roda e quanto custa por quadro.
- A causa dos erros (abrir as fotos erradas e classificá-las) não foi analisada, por decisão da Joyce.
- 248 das 7.750 fotos de validação têm cópia exata no treino (efeito não medido); os rótulos têm ruído (kappa de 0,22 a 0,42).
- O bootstrap mede só o sorteio das fotos de validação, e há vários intervalos de 95% ao mesmo tempo.
